# 🐳 Day 150 — Docker Basics + Containerization
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 150 (Month 8, Week 3, Day 5) |
| **Topic** | Docker fundamentals · Dockerfile · docker-compose · Containerizing Streamlit + FastAPI |
| **Dataset** | FreelanceHub India (500 rows, seed=141) — carried from Day 149 |
| **Deliverables** | `day149_api/Dockerfile` · `day149_app/Dockerfile` · `docker-compose.yml` · `.dockerignore` (×2) · `Day150_Docker_Concepts.ipynb` |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 🗺️ Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals | 80/80 + 10★ ✅ |
| 148 | FastAPI + DB + Auth | 80/80 + 10★ ✅ |
| 149 | Streamlit ↔ FastAPI Full-Stack Integration | 80/80 + 10★ ✅ |
| **150** | **Docker Basics + Containerization** | **← Today** |

---

## 🏗️ What You Are Building Today

```
Day149 project structure (what you already have)
├── day149_api/
│   ├── main.py           ← FastAPI app
│   ├── freelancehub.db   ← SQLite database
│   └── model.joblib      ← trained Random Forest
└── day149_app/
    └── streamlit_app.py  ← Streamlit frontend

Day 150 adds:
├── day149_api/
│   ├── Dockerfile        ← ⭐ NEW
│   ├── requirements.txt  ← ⭐ NEW
│   └── .dockerignore     ← ⭐ NEW
├── day149_app/
│   ├── Dockerfile        ← ⭐ NEW
│   ├── requirements.txt  ← ⭐ NEW
│   └── .dockerignore     ← ⭐ NEW
└── docker-compose.yml    ← ⭐ NEW (root level)
```

> **Freelance angle:** Right now your app runs only on YOUR machine.  
> Docker packages everything (code + Python version + libraries) into one box  
> that runs identically on any computer, server, or cloud VPS.  
> This is what separates a "local prototype" from a "deployable product."


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** This is the source dataset — reference only.
> Day 150 carries forward the FreelanceHub India dataset from Day 149.


In [ ]:
import pandas as pd
import numpy as np

# ─── FreelanceHub India Dataset (seed=141, 500 rows) ───────────────────────
np.random.seed(141)
n = 500

categories  = ['Web Dev','ML/AI','Data Analysis','Mobile Dev','UI/UX','SEO/Content']
skills      = ['Python','JavaScript','SQL','React','Figma','TensorFlow','Power BI','Flutter']
cities      = ['Mumbai','Delhi','Bangalore','Hyderabad','Chennai','Pune','Kolkata']
levels      = ['Junior','Mid','Senior','Expert']
statuses    = ['Completed','In Progress','Cancelled']
platforms   = ['Upwork','Freelancer','Fiverr','Toptal']

df = pd.DataFrame({
    'project_id'    : range(1, n+1),
    'category'      : np.random.choice(categories, n),
    'skill'         : np.random.choice(skills, n),
    'city'          : np.random.choice(cities, n),
    'freelancer_lvl': np.random.choice(levels, n),
    'status'        : np.random.choice(statuses, n, p=[0.65, 0.25, 0.10]),
    'hourly_rate'   : np.round(np.random.uniform(800, 3500, n), 2),
    'project_value' : np.round(np.random.uniform(5000, 120000, n), 2),
    'duration_days' : np.random.randint(7, 120, n),
    'client_rating' : np.where(
        np.random.random(n) > 0.05,
        np.round(np.random.uniform(3.0, 5.0, n), 1),
        np.nan
    ),
    'platform'      : np.random.choice(platforms, n),
    'high_value'    : None  # will be computed
})
df['high_value'] = (df['project_value'] > 50000).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head(3))


---
## 📚 Section 2 — Concept Notes

### 2.1 What is Docker?

Docker is a platform that packages your application and all its dependencies into a
**container** — an isolated, self-contained unit that runs identically everywhere.

```
Without Docker                      With Docker
─────────────────────────           ─────────────────────────────────
"It works on my machine"            Same container image runs on:
Dev: Python 3.11 + lib A v1.2       ├── Your laptop
Prod: Python 3.9 + lib A v0.9       ├── Your client's server
Result: crashes in production       ├── AWS / Google Cloud / Hetzner
                                    └── CI/CD pipelines
```

### 2.2 Core Docker Concepts

| Term | What It Is | Analogy |
|---|---|---|
| **Image** | Blueprint — frozen snapshot of app + OS + libs | Recipe |
| **Container** | Running instance of an image | Cooked dish from recipe |
| **Dockerfile** | Instructions to build an image | The recipe itself |
| **docker-compose** | Tool to run multiple containers together | Kitchen with multiple dishes |
| **Registry** | Storage for images (Docker Hub, GHCR) | Cookbook library |

### 2.3 Dockerfile Instructions (All 6 You Need Today)

| Instruction | Purpose | Example |
|---|---|---|
| `FROM` | Base image (OS + runtime) | `FROM python:3.11-slim` |
| `WORKDIR` | Set working directory inside container | `WORKDIR /app` |
| `COPY` | Copy files from host → container | `COPY requirements.txt .` |
| `RUN` | Execute a shell command at build time | `RUN pip install -r requirements.txt` |
| `EXPOSE` | Document which port the app uses | `EXPOSE 8000` |
| `CMD` | Default command to run the container | `CMD ["uvicorn", "main:app", ...]` |

### 2.4 Why `python:3.11-slim`?

```
python:3.11          # full image  ~900 MB — includes compilers, dev tools
python:3.11-slim     # slim image  ~120 MB — only essentials
python:3.11-alpine   # tiny image   ~50 MB — minimal Linux (sometimes causes build issues)
```

**Rule of thumb:** Use `-slim` for production. It's small, stable, and pip-compatible.

### 2.5 The Critical `--host 0.0.0.0` Flag

```
Inside a container, "localhost" means the CONTAINER itself — not your laptop.

uvicorn main:app --host 127.0.0.1  # ❌ Only reachable from inside the container
uvicorn main:app --host 0.0.0.0    # ✅ Reachable from outside (your browser, other containers)

Same rule applies to Streamlit:
streamlit run app.py --server.address=0.0.0.0  # ✅ Required for Docker
```

### 2.6 docker-compose.yml — The Orchestrator

docker-compose runs multiple containers as a **single application**:

```yaml
services:
  api:          # Service name = hostname inside Docker network
    build: ./day149_api
    ports:
      - "8000:8000"    # "HOST:CONTAINER"
  
  frontend:
    build: ./day149_app
    ports:
      - "8501:8501"
    depends_on:
      - api     # Start api first, then frontend
    environment:
      - API_BASE_URL=http://api:8000   # "api" resolves inside Docker network!
```

> **Key insight:** Inside docker-compose, services talk to each other using service names as hostnames.  
> `http://api:8000` works because Docker creates a private network where `api` = the API container's IP.

### 2.7 .dockerignore — What Not to Copy

Just like `.gitignore`, `.dockerignore` tells Docker what to skip:
```
__pycache__/    ← compiled Python bytecode (container regenerates it)
*.db            ← database (should use volume, not bake into image)
.env            ← secrets! never copy into image
venv/           ← heavy; container installs its own from requirements.txt
.git/           ← source control history not needed in runtime image
*.ipynb         ← teaching notebooks, not app code
```

### 2.8 Essential Docker Commands (Day 150 Reference)

```bash
# Build image from Dockerfile in current directory
docker build -t myapp:latest .

# Run a container
docker run -p 8000:8000 myapp:latest

# Run entire docker-compose stack
docker compose up

# Run in background (detached mode)
docker compose up -d

# Stop everything
docker compose down

# View running containers
docker ps

# View logs for a service
docker compose logs api

# Rebuild after code changes
docker compose up --build
```


---
## 🏋️ Section 3 — Practice Guide (80 pts)

### Folder Structure to Create

```
Month8/
├── day149_api/           ← (already exists from Day 149)
│   ├── main.py
│   ├── Dockerfile        ← Task 1 (write this)
│   ├── requirements.txt  ← Task 2 (write this)
│   └── .dockerignore     ← Task 4 (write this)
├── day149_app/           ← (already exists from Day 149)
│   ├── streamlit_app.py
│   ├── Dockerfile        ← Task 1 (write this)
│   ├── requirements.txt  ← Task 2 (write this)
│   └── .dockerignore     ← Task 4 (write this)
└── docker-compose.yml    ← Task 3 (write this — at root level)
```

> **Note:** If you don't have the Day 149 files, create minimal stubs  
> (`main.py` with just `from fastapi import FastAPI; app = FastAPI()` and  
> `streamlit_app.py` with just `import streamlit as st; st.write("Hello")`)  
> — the Docker concepts are identical regardless of app complexity.

---

### Task 1 — FastAPI Dockerfile (20 pts)

Write `day149_api/Dockerfile`. Requirements:
- [ ] Base image: `python:3.11-slim`
- [ ] WORKDIR: `/app`
- [ ] Copy `requirements.txt` first (before other files — **why does this matter?**)
- [ ] RUN pip install with `--no-cache-dir` flag
- [ ] Copy remaining app files (`. .`)
- [ ] EXPOSE port `8000`
- [ ] CMD: run uvicorn on `main:app`, host `0.0.0.0`, port `8000`

Write `day149_app/Dockerfile`. Requirements (same structure, different CMD):
- [ ] Same base image, WORKDIR, COPY+RUN pattern
- [ ] EXPOSE port `8501`
- [ ] CMD: run streamlit on `streamlit_app.py`, port `8501`, address `0.0.0.0`
- [ ] Streamlit CMD must also include `--server.headless=true` (why?)

**Written Q (5 pts):** Why do we `COPY requirements.txt .` and `RUN pip install`  
BEFORE `COPY . .`? Explain in one sentence using the word "cache."

---

### Task 2 — requirements.txt Files (10 pts)

Write `day149_api/requirements.txt` with exact packages (no version pins required):
- fastapi, uvicorn[standard], pandas, scikit-learn, joblib, pydantic

Write `day149_app/requirements.txt`:
- streamlit, requests, pandas, plotly

---

### Task 3 — docker-compose.yml (25 pts)

Write `docker-compose.yml` at the root (parent of `day149_api/` and `day149_app/`).

Requirements:
- [ ] version: `'3.8'`
- [ ] Two services: `api` and `frontend`
- [ ] `api` service:
  - build context: `./day149_api`
  - ports: `"8000:8000"`
  - environment: `API_KEY=freelance_secret_key`
- [ ] `frontend` service:
  - build context: `./day149_app`
  - ports: `"8501:8501"`
  - `depends_on: [api]`
  - environment:
    - `API_BASE_URL=http://api:8000`
    - `API_KEY=freelance_secret_key`

**Written Q (5 pts):** In the `frontend` environment, why is the URL  
`http://api:8000` and NOT `http://localhost:8000`?

---

### Task 4 — .dockerignore Files (5 pts)

Create `.dockerignore` inside BOTH `day149_api/` and `day149_app/` with at least 5 patterns.  
Must include: `__pycache__/`, `*.db`, `.env`, `venv/`, `.git/`

---

### Task 5 — NRA Insight (10 pts)

In a new file `Day150_NRA.md`, write one NRA insight about your Docker deployment.

Format:
```
**Number:** [cite 2+ specific numbers from your Docker setup]
**Reason:** [why containerization solves a real client problem]
**Action:** [specific next step — name a platform, cost, timeline]
```

Example of a WEAK action (0 pts for Action): "Deploy to the cloud."  
Example of a STRONG action (5/5 pts): "Deploy to Hetzner CX11 (₹500/month, 2 vCPU, 4GB RAM)
using `docker compose up -d` — stack is production-ready in under 10 minutes."

---

### ⭐ Bonus Task (10★) — Multi-Stage Build

Rewrite ONE of the Dockerfiles using a **multi-stage build** pattern:

```dockerfile
# Stage 1: builder
FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

# Stage 2: runtime (smaller final image)
FROM python:3.11-slim
WORKDIR /app
COPY --from=builder /install /usr/local
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

Written answer: What is the benefit of this pattern vs a single-stage build?


---
## 📊 Section 4 — Scoring Rubric (80 pts + 10★)

| Task | Sub-task | Points |
|---|---|---|
| **T1: FastAPI Dockerfile** | FROM python:3.11-slim | 2 |
| | WORKDIR /app | 2 |
| | COPY requirements.txt . (before COPY . .) | 3 |
| | RUN pip install --no-cache-dir | 3 |
| | EXPOSE 8000 | 2 |
| | CMD with 0.0.0.0 host | 3 |
| | **Written Q — cache explanation** | **5** |
| | _T1 Subtotal_ | _20_ |
| **T1: Streamlit Dockerfile** | All 6 instructions correct | 8 |
| | Correct port (8501) | 2 |
| | CMD includes --server.headless=true | 5 |
| | CMD includes --server.address=0.0.0.0 | 5 |
| | _T1 Streamlit Subtotal_ | _20_ |
| **T2: requirements.txt** | FastAPI: all 6 packages present | 5 |
| | Streamlit: all 4 packages present | 5 |
| | _T2 Subtotal_ | _10_ |
| **T3: docker-compose.yml** | version: '3.8' | 2 |
| | api service: build + ports | 5 |
| | api service: environment | 3 |
| | frontend service: build + ports | 5 |
| | frontend service: depends_on | 3 |
| | frontend environment (both vars) | 2 |
| | **Written Q — localhost vs api** | **5** |
| | _T3 Subtotal_ | _25_ |
| **T4: .dockerignore** | Both files present, ≥5 patterns each | 5 |
| | _T4 Subtotal_ | _5_ |
| **T5: NRA Insight** | Number: 2+ specific numbers cited | 3 |
| | Reason: solves a real deployment problem | 3 |
| | Action: specific platform + cost + timeline | 4 |
| | _T5 Subtotal_ | _10_ |
| **⭐ Bonus** | Multi-stage build correct + written answer | 10★ |
| | **TOTAL** | **80 + 10★** |

---

## 🎙️ Interview / Client Framing

**Q: "Why Docker instead of just giving the client a requirements.txt?"**

**A:** "A `requirements.txt` tells a machine *what* to install but not *which Python version*,
not the OS, not system-level dependencies like `libgomp` for scikit-learn.
I've seen models fail in production because the client had Python 3.9 while I built on 3.11.
Docker packages the entire runtime stack — OS layer, Python version, libraries — into one image.
`docker compose up` is all the client needs. I use it for all client deliverables now;
it's the difference between a prototype and a deployable product."

---

## 🔑 Key Takeaway — Day 150

> **COPY requirements.txt before COPY . .**  
> This one-line ordering decision saves minutes on every rebuild.  
> Docker caches layers. If requirements.txt hasn't changed, the pip install layer  
> is reused. If you copy everything first, ANY code change invalidates the pip cache  
> and forces a full reinstall. This is the single most common Docker performance mistake.
